In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(readr)

In [ ]:
#-- Load outcomes
combo <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Combination_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
aug <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Augmentation_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_90days_noBIP_noPsychotic.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence)
  ) 

#== Identify drug-class with at least 2,000 people
classes <- outcome %>%
  group_by(Category.treatment) %>%
  summarise(
    count = n_distinct(person_id)
  ) %>%
  filter(count >= 2000) %>%
  pull(Category.treatment)
classes <- c("Pooled", classes)
classes <- c("Pooled")

#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
pheno <- pheno %>% 
    mutate(Income_quantitative =
          case_when(Income == ">200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "<=25k" ~ 1,
                   TRUE ~ NA_integer_))


first_dispense <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/First_Dispense.csv")
pheno <- pheno %>%
    left_join(first_dispense, by = "person_id")

#-- Load PGS
pgs <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/all_traits_PGS.csv")
colnames(pgs)


#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))

anc_df_clean <- anc_df %>%
  mutate(pca_clean = str_remove_all(pca_features, "\\[|\\]")) 
anc_df_clean2 <- anc_df_clean %>%
  separate(pca_clean, into = paste0("PC", 1:16), sep = ",\\s*", convert = TRUE)
pc <- anc_df_clean2 %>% select(research_id, ancestry_pred, PC1:PC16)

pgs_pc <- pgs %>% left_join(pc, by = c("IID" = "research_id"))
eur_pgs_pc <- pgs_pc %>% 
    filter(ancestry_pred == "eur")

# Define all PGS columns to standardize
pgs_cols <- c("ADHD_01_PGS", "Migraine_01_PGS", "Insomnia_01_PGS", 
              "MDD_04_PGS", "SCZ_03_PGS", "BIP_2025_PGS", "BMI_LOO_PGS")

# Standardize each using EUR means and SDs
for (col in pgs_cols) {
  meu <- mean(eur_pgs_pc[[col]], na.rm = TRUE)
  sd <- sd(eur_pgs_pc[[col]], na.rm = TRUE)
  
  new_col <- paste0(col, "_std")
  pgs_pc[[new_col]] <- (pgs_pc[[col]] - meu) / sd
}

In [ ]:
#-- Function to simplify data set to first switch away from DrugClass
extract_outcomes <- function(class){
  
  if (class == "Pooled") {
    #-- First switch
    first_switch <- outcome %>%
      filter(FinalClassification.treatment == "Switching") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
  
    #-- First continuation (among non-switchers)
    C <- outcome %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(FinalClassification.treatment == "Continuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
      
    #-- Filter continuers to have no TRD, augmentation or combination
    C_clean <- C %>%
      filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
      filter(!person_id %in% combo$person_id[combo$Case == 1])
    
    #-- First discontinuation (among non-switchers AND non-continuers)
    ED <- outcome %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(!person_id %in% C$person_id) %>%
      filter(FinalClassification.treatment == "Discontinuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
      
     ED_clean <- ED %>%
      filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
      filter(!person_id %in% combo$person_id[combo$Case == 1])
    
  } else {
    #-- First switch from this class to a different class
    first_switch <- outcome %>%
      filter(Category.treatment == class) %>%
      filter(Category.other != class) %>%
      filter(FinalClassification.treatment == "Switching") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
    
    #-- First continuation of this class (among non-switchers)
    C <- outcome %>%
      filter(Category.treatment == class) %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(FinalClassification.treatment == "Continuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
      
    #-- Filter continuers to have no TRD, augmentation or combination
    C_clean <- C %>%
      filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
      filter(!person_id %in% combo$person_id[combo$Case == 1])
    
    #-- First discontinuation of this class (among non-switchers AND non-continuers)
    ED <- outcome %>%
      filter(Category.treatment == class) %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(!person_id %in% C$person_id) %>%
      filter(FinalClassification.treatment == "Discontinuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
      
    ED_clean <- ED %>%
      filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
      filter(!person_id %in% combo$person_id[combo$Case == 1])
  }
  
  #== Input data 
  dat <- rbind(first_switch, C_clean, ED_clean)
  print(table(dat$FinalClassification.treatment))
  
  #== Join with phenotypes
  both <- dat %>%
    left_join(pheno, by = "person_id") %>%
    mutate(
      age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
    )
  
  #=== Join with genetic info
  input_data <- both %>% 
    left_join(pgs_pc, by = c("person_id" = "IID"))
  
  #-- Set Continuation as the reference
  input_data$FinalClassification.treatment <- factor(input_data$FinalClassification.treatment, levels = c("Continuation", "Discontinuation", "Switching"))
  input_data$FinalClassification.treatment <- relevel(input_data$FinalClassification.treatment, ref = "Continuation")
  
  return(input_data)
}

In [ ]:
#-- Pre-compute all extract_outcomes() calls
cat("Pre-loading data...\n")
class_data <- list()
for (class in classes) {
  cat("Loading class:", class, "\n")
  flush.console()
  class_data[[class]] <- extract_outcomes(class)
}
cat("Data loaded!\n\n")

In [ ]:
# normal logistic regression
model_function <- function(input_data, independent) {
  
  print(independent)
  
  #-- Filter for complete cases (include PCs for residualisation, not modelling)
  input_data <- input_data %>%
    select(FinalClassification.treatment, age_at_first_AD, sex_at_birth, Income_quantitative, Education_level,
           PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, PC11, PC12, PC13, PC14, PC15, PC16,
           !!independent) %>%
    filter(complete.cases(.))
  
  #-- Check if any data remains after filtering
  if (nrow(input_data) == 0) {
    message("No complete cases remaining")
    return("Incomplete")
  }
  
  #-- Residualise PGS on global PCs
  pc_formula <- as.formula(paste(independent, "~ PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 +
                                  PC11 + PC12 + PC13 + PC14 + PC15 + PC16"))
  input_data$pgs_residualised <- residuals(lm(pc_formula, data = input_data))
  
  #-- Outcome prevalence (for the diagnostic gate only)
  outcome_counts <- input_data %>%
    group_by(FinalClassification.treatment) %>%
    summarise(n = n(), .groups = "drop")
  
  n_continuation <- outcome_counts %>% filter(FinalClassification.treatment == "Continuation") %>% pull(n)
  n_ed           <- outcome_counts %>% filter(FinalClassification.treatment == "Discontinuation") %>% pull(n)
  n_switching    <- outcome_counts %>% filter(FinalClassification.treatment == "Switching") %>% pull(n)
  
  if (length(n_continuation) == 0) n_continuation <- 0
  if (length(n_ed) == 0) n_ed <- 0
  if (length(n_switching) == 0) n_switching <- 0
  
  #-- Formula uses residualised PGS, no PCs in outcome model
  glm_formula <- as.formula(
    "outcome_binary ~ age_at_first_AD + sex_at_birth + Income_quantitative + Education_level + pgs_residualised"
  )
  
  #-- Helper to run one binary logistic regression and extract results
  run_glm <- function(data, comparison_label, ref_label, case_label) {
    
    model  <- glm(glm_formula, data = data, family = binomial())
    
    coefs  <- coef(model)
    ses    <- sqrt(diag(vcov(model)))
    lower  <- coefs - 1.96 * ses
    upper  <- coefs + 1.96 * ses
    p_vals <- summary(model)$coefficients[, 4]
    
    #-- Rename pgs_residualised back to the original PGS name for interpretability
    names(coefs)[names(coefs) == "pgs_residualised"]  <- independent
    names(lower)[names(lower) == "pgs_residualised"]  <- independent
    names(upper)[names(upper) == "pgs_residualised"]  <- independent
    names(p_vals)[names(p_vals) == "pgs_residualised"] <- independent
    names(ses)[names(ses) == "pgs_residualised"]      <- independent
    
    #-- Ns computed from the data actually being modelled
    n_ref  <- sum(data$outcome_binary == 0)
    n_case <- sum(data$outcome_binary == 1)
    
    data.frame(
      outcome_comparison = comparison_label,
      variable           = names(coefs),
      coef               = coefs,
      odds_ratio         = exp(coefs),
      se                 = ses,
      p_value            = p_vals,
      lower_95           = exp(lower),
      upper_95           = exp(upper),
      N_Total            = nrow(data),
      N_Reference        = n_ref,
      N_Case             = n_case,
      Reference_Group    = ref_label,
      Case_Group         = case_label,
      stringsAsFactors   = FALSE,
      row.names          = NULL
    )
  }
  
  #-- Switching vs Continuation
  data_sw <- input_data %>%
    filter(FinalClassification.treatment %in% c("Continuation", "Switching")) %>%
    mutate(outcome_binary = as.integer(FinalClassification.treatment == "Switching"))
  
  #-- Discontinuation vs Continuation
  data_ed <- input_data %>%
    filter(FinalClassification.treatment %in% c("Continuation", "Discontinuation")) %>%
    mutate(outcome_binary = as.integer(FinalClassification.treatment == "Discontinuation"))
  
  #-- Gate each comparison independently (need >= 20 in BOTH arms)
  results_list <- list()
  
  if (sum(data_sw$outcome_binary == 1) >= 20 && sum(data_sw$outcome_binary == 0) >= 20) {
    results_list[["sw"]] <- run_glm(
      data_sw,
      "Switching vs Continuation",
      "Continuation",
      "Switching"
    )
  } else {
    message(sprintf(
      "Skipping Switching vs Continuation: Cont=%d, Switch=%d",
      sum(data_sw$outcome_binary == 0),
      sum(data_sw$outcome_binary == 1)
    ))
  }
  
  if (sum(data_ed$outcome_binary == 1) >= 20 && sum(data_ed$outcome_binary == 0) >= 20) {
    results_list[["ed"]] <- run_glm(
      data_ed,
      "Discontinuation vs Continuation",
      "Continuation",
      "Discontinuation"
    )
  } else {
    message(sprintf(
      "Skipping Discontinuation vs Continuation: Cont=%d, Discont=%d",
      sum(data_ed$outcome_binary == 0),
      sum(data_ed$outcome_binary == 1)
    ))
  }
  
  #-- If nothing ran, return "Incomplete" so the caller's `is.data.frame` check skips it
  if (length(results_list) == 0) {
    return("Incomplete")
  }
  
  results <- bind_rows(results_list)
  return(results)
}

In [ ]:
#-- Select phenotypes to be associated with 
predictors <- c("ADHD_01_PGS_std", "Migraine_01_PGS_std", "Insomnia_01_PGS_std", 
              "MDD_04_PGS_std", "SCZ_03_PGS_std", "BIP_2025_PGS_std","BMI_LOO_PGS_std")

In [ ]:
#-- Load Switching Classifications and loop through each condition combination
population_groups <- c("afr", "eur", "amr")
clinical_groups = c("Full", "MDD", "noMDD", "ANX", "CIDI-MDD", "CIDI-noMDD")
classes <- c("Pooled")
all_list <- list() 
    
for (clinical in clinical_groups) {
  
  cat("Clinical:", clinical, "\n")
    
  #======== Class-level analysis
  class_results_list <- list()
  indexes <- character(0)
  
  for (class in classes){
    
    cat("Class:", class, "\n")
    final <- class_data[[class]]
    
    #== Filter for clinical group of interest
    if (clinical == "MDD") {
      input_data <- final %>%
        filter(Depression == 1)
    } else if (clinical == "noMDD") {
      input_data <- final %>%
        filter(Depression == 0)
    } else if (clinical == "ANX") {
        input_data <- final %>%
            filter(Anxiety == 1)
    } else if (clinical == "CIDI-MDD") {
        input_data <- final %>%
            filter(MDD_Incl == 1)
    } else if (clinical == "CIDI-noMDD") {
        input_data <- final %>%
            filter(MDD_Incl == 0)
    } else {
        input_data <- final
    }
      
    #== Filter for population group
    for (population in population_groups) {
        
        cat("Ancestry:", population, "\n")
        if (population != "eur" & class != "Pooled") next
    
        input_data_anc <- input_data %>%
            filter(ancestry_pred == population)
        
        results_list <- list()
        
        #== Loop through each genetic predictor
        for (independent in predictors){
          
           multinom_results <- tryCatch(
            {
              model_function(input_data_anc, independent)
            },
            error = function(e) {
              message("Error for ", independent, ":", e$message)
              return("Error")
             }
            )
      
            if (!is.data.frame(multinom_results)) next
      
            multinom_results <- multinom_results %>%
                mutate(
                  GeneticMarker = independent,
                  ClinicalGroup = clinical,
                  Population = population,
                  Class = class
                )
            results_list[[independent]] <- multinom_results
        }
        
        pop_results_df <- do.call(rbind, results_list)
        if (is.null(class_results_list[[class]])) {
          class_results_list[[class]] <- pop_results_df
        } else {
          class_results_list[[class]] <- rbind(class_results_list[[class]], pop_results_df)
        }
        indexes <- names(results_list)
    }
  }
  
  #== Combine results
  results <- do.call(rbind, lapply(class_results_list, data.frame))
  rownames(results) <- NULL
  
  #== Multiple testing correction
    results_format <- results %>%
      select(Population, ClinicalGroup, Class, outcome_comparison, GeneticMarker, 
             N_Total, N_Reference, N_Case, 
             variable, odds_ratio, lower_95, upper_95, p_value)
  
    #-- Compute Bonferroni on PGS rows only
    bonf_pgs <- results_format %>%
      filter(variable == GeneticMarker) %>%
      group_by(Population, ClinicalGroup, Class, outcome_comparison) %>%
      mutate(Bonf_P = pmin(p_value * n_distinct(GeneticMarker), 1)) %>%
      ungroup() %>%
      select(Population, ClinicalGroup, Class, outcome_comparison, GeneticMarker, variable, Bonf_P)

    #-- Join back to full results
    results_terms <- results_format %>%
      left_join(
        bonf_pgs,
        by = c("Population", "ClinicalGroup", "Class", "outcome_comparison", "GeneticMarker", "variable")
      )
  
  #== Format results
  results_format_rounded <- results_terms %>%
      mutate(
        across(c(odds_ratio, lower_95, upper_95), ~ signif(.x, 4)),
        p_value_num = p_value,
        Bonf_P_num  = Bonf_P,
        p_value_display = case_when(
          p_value < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(p_value, format = "e", digits = 2)
        ),
        Bonf_P_display = case_when(
          Bonf_P < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(Bonf_P, format = "e", digits = 2)
        )
      )
  all_list[[clinical]] <- results_format_rounded
}

In [ ]:
results_df <- bind_rows(all_list)
results_df <- results_df %>%
    mutate(
        GeneticMarker = case_when(
        GeneticMarker == "ADHD_01_PGS_std" ~ "ADHD",
        GeneticMarker == "Migraine_01_PGS_std" ~ "Migraine",
        GeneticMarker == "Insomnia_01_PGS_std" ~ "Insomnia",
        GeneticMarker == "ANX_2025_PGS_std" ~ "Anxiety",
        GeneticMarker == "MDD_04_PGS_std" ~ "Depression",
        GeneticMarker == "SCZ_03_PGS_std" ~ "Schizophrenia",
        GeneticMarker == "BIP_2025_PGS_std" ~ "Bipolar Disorder",
        GeneticMarker == "BMI_LOO_PGS_std" ~ "BMI",
        TRUE ~ NA_character_)) %>%
    mutate(variable = case_when(
        variable == "ADHD_01_PGS_std" ~ "ADHD",
        variable == "Migraine_01_PGS_std" ~ "Migraine",
        variable == "Insomnia_01_PGS_std" ~ "Insomnia",
        variable == "ANX_2025_PGS_std" ~ "Anxiety",
        variable == "MDD_04_PGS_std" ~ "Depression",
        variable == "SCZ_03_PGS_std" ~ "Schizophrenia",
        variable == "BIP_2025_PGS_std" ~ "Bipolar Disorder",
        variable == "BMI_LOO_PGS_std" ~ "BMI",
        TRUE ~ variable)) %>%
    mutate(Population = case_when(
        Population == "afr" ~ "African",
        Population == "eur" ~ "European",
        Population == "amr" ~ "Latino/Mixed-American",
        TRUE ~ NA_character_)) %>%
    mutate(ClinicalGroup = case_when(
        ClinicalGroup == "Full" ~ "Full Cohort",
        ClinicalGroup == "MDD" ~ "Recorded Depression",
        ClinicalGroup == "noMDD" ~ "No Recorded Depression",
        ClinicalGroup == "ANX" ~ "Recorded Anxiety",
        TRUE ~ ClinicalGroup))


pop_order <- c("European", "African", "Latino/Mixed-American")

clinical_order <- c(
  "Full Cohort",
  "Recorded Depression",
  "No Recorded Depression",
  "Recorded Anxiety",
    "CIDI-MDD",
    "CIDI-noMDD"
)

results_df <- results_df %>%
  rename(Outcome = outcome_comparison,
        PGS = GeneticMarker) %>%
  mutate(
    Population = factor(Population, levels = pop_order),
    ClinicalGroup = factor(ClinicalGroup, levels = clinical_order)
  ) %>%
  arrange(Population, ClinicalGroup, Class, PGS, Outcome)

results_df_pooled <- results_df %>%
    filter(Class == "Pooled") %>%
    select(-Class) %>%
    filter(Population == "European" | !ClinicalGroup %in% c("CIDI-MDD", "CIDI-noMDD")) %>%
    filter(Outcome == "Discontinuation vs Continuation")

In [ ]:
#-- Only save the pooled outcome results
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx")
removeWorksheet(wb, "Table3_Dis_Sens")
addWorksheet(wb, "Table3_Dis_Sens")
writeData(wb, "Table3_Dis_Sens", results_df_pooled)
saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", overwrite=TRUE)

In [ ]:
library(readxl)
library(dplyr)
library(tidyverse)
library(stringr)
library(cowplot)

#-- Paths
aou_path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx"

#-- Common filtering + recoding for both cohorts
prep <- function(raw, p_col = "Bonf_P") {
  raw %>%
    filter(!variable %in% c("(Intercept)", "sex_at_birthMale", "sex_at_birth", "age_at_first_AD")) %>%
    filter(!str_detect(variable, "PC")) %>%
    filter(!str_detect(variable, "Income")) %>%
    filter(!str_detect(variable, "Education_level")) %>%
    filter(Population == "European") %>%
    filter(ClinicalGroup != "Recorded Anxiety") %>%
    filter(variable == PGS) %>%
    mutate(
      odds_ratio    = as.numeric(odds_ratio),
      lower_95      = as.numeric(lower_95),
      upper_95      = as.numeric(upper_95),
      SigFill       = if_else(.data[[p_col]] < 0.05, ClinicalGroup, "NotSig")
    )
}

aou_dat <- prep(read_excel(aou_path, sheet = "Table3_Dis_Sens"))

#-- PGS order from AoU Discontinuation, Depression subset
pgs_order <- aou_dat %>%
  filter(Outcome == "Discontinuation vs Continuation", ClinicalGroup == "Recorded Depression") %>%
  arrange(odds_ratio) %>%
  pull(PGS) %>% unique()

aou_dat$PGS <- factor(aou_dat$PGS, levels = pgs_order)

cg_levels <- rev(c("Recorded Depression", "Full Cohort", "No Recorded Depression", "CIDI-MDD", "CIDI-noMDD"))
aou_dat$ClinicalGroup <- factor(aou_dat$ClinicalGroup, levels = cg_levels)

#-- Tag and combine
aou_tagged <- aou_dat %>% mutate(Cohort = "All of Us")
combined   <- bind_rows(aou_tagged)

#-- Build "Cohort\nDepression: N=...\nNo Depression: N=...\nFull Cohort: N=..." per cohort × outcome
make_facet_label <- function(d) {
  ns <- d %>%
    select(ClinicalGroup, N_Case, N_Total) %>%
    distinct() %>%
    arrange(factor(ClinicalGroup, levels = c("Recorded Depression", "Full Cohort", "No Recorded Depression", "CIDI-MDD", "CIDI-noMDD")))
  paste0(unique(d$Cohort), "\n",
         paste(sprintf("%s: N=%s  (%s%%)", ns$ClinicalGroup,
                       format(ns$N_Total, big.mark = ","),
                       round((ns$N_Case / ns$N_Total) * 100, 0)),
               collapse = "\n"))
}

combined <- combined %>%
  group_by(Cohort, Outcome) %>%
  mutate(FacetLabel = make_facet_label(cur_data_all())) %>%
  ungroup() %>%
  mutate(FacetLabel = factor(
    FacetLabel,
    levels = unique(FacetLabel[order(match(Cohort, c("All of Us", "Lifelines")))])
  ))

#-- Aesthetics
fill_colors <- c(
  "No Recorded Depression" = "#2CA58D80",
  "Recorded Depression"    = "#D11149",
  "Full Cohort"   = "#6610F280",
       "CIDI-noMDD"       = "#2E6F9580",
  "CIDI-MDD"      = "#E0A526"
)
pos <- position_dodge(width = 0.6)

#-- Row builder: one outcome, faceted by cohort
make_row <- function(dat, outcome, row_title, show_legend = FALSE) {
  d <- dat %>% filter(Outcome == outcome)
  ggplot(d, aes(x = odds_ratio, y = PGS, group = ClinicalGroup)) +
    geom_vline(xintercept = 1, linetype = "dashed") +
    geom_errorbarh(aes(xmin = lower_95, xmax = upper_95, colour = ClinicalGroup),
                   height = 0, position = pos) +
    geom_point(aes(fill = SigFill, colour = ClinicalGroup),
               shape = 21, size = 4, stroke = 0.5, position = pos) +
    scale_color_manual(values = fill_colors) +
    scale_fill_manual(values = c(fill_colors, NotSig = "white"), guide = "none") +
    guides(color = guide_legend(nrow = 3)) +
    facet_wrap(~ FacetLabel, nrow = 1, scales = "free_x") +
    labs(x = "Odds ratio with 95% CI,\nRef = Continuation",
         y = "PGS", title = row_title, color = "Diagnostic\nSubset") +
    theme_minimal() +
    theme(
      axis.title       = element_text(color = "black", face = "bold"),
      plot.title       = element_text(face = "bold", size = 14, hjust = 0.5),
      strip.text       = element_text(face = "bold", size = 12),
      strip.background = element_rect(fill = "white", color = "black"),
      axis.text        = element_text(color = "black"),
      axis.text.y      = element_text(size = 13, color = "black"),
      panel.background = element_rect(fill = "white", colour = NA),
      plot.background  = element_rect(fill = "white", colour = NA),
      plot.margin      = margin(t = 20, r = 5, b = 5, l = 5),
      legend.title     = element_text(face = "bold"),
      legend.position  = if (show_legend) "bottom" else "none",
      legend.text      = element_text(size = 10)
    )
}

#-- Build rows and combine
p_disc   <- make_row(combined, "Discontinuation vs Continuation", "Discontinuation", show_legend = TRUE)
p_disc


In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Pooled_Disc_Sens_PGS.png",
       plot   = p_disc,
       width  = 7,
       height = 7,
       device = "png",
       units  = "in")